### PHASE 1 DATA PREPARATION AND CLEANING 


1. Prepare & Match Datasets

In [36]:
import pandas as pd
import re

#Load the datasets
fuel_df = pd.read_csv('FuelConsumption.csv')     # 2014 car fuel/emission data
cars_df = pd.read_csv('cars_2025.csv')            # 2025 Malaysia car registration data

#show only the first 5 rows of fuel data
fuel_df.info()
fuel_df.describe()
fuel_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   MODELYEAR                 1067 non-null   int64  
 1   MAKE                      1067 non-null   object 
 2   MODEL                     1067 non-null   object 
 3   VEHICLECLASS              1067 non-null   object 
 4   ENGINESIZE                1067 non-null   float64
 5   CYLINDERS                 1067 non-null   int64  
 6   TRANSMISSION              1067 non-null   object 
 7   FUELTYPE                  1067 non-null   object 
 8   FUELCONSUMPTION_CITY      1067 non-null   float64
 9   FUELCONSUMPTION_HWY       1067 non-null   float64
 10  FUELCONSUMPTION_COMB      1067 non-null   float64
 11  FUELCONSUMPTION_COMB_MPG  1067 non-null   int64  
 12  CO2EMISSIONS              1067 non-null   int64  
dtypes: float64(4), int64(4), object(5)
memory usage: 108.5+ KB


,MODELYEAR,MAKE,MODEL,VEHICLECLASS,ENGINESIZE,CYLINDERS,TRANSMISSION,FUELTYPE,FUELCONSUMPTION_CITY,FUELCONSUMPTION_HWY,FUELCONSUMPTION_COMB,FUELCONSUMPTION_COMB_MPG,CO2EMISSIONS
0,2014,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,2014,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,2014,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,2014,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,2014,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


In [79]:
#show only the first 5 rows of cars data
cars_df.info()
cars_df.describe()
cars_df.head()


<class 'pandas.core.frame.DataFrame'>
Index: 48437 entries, 0 to 263576
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   date_reg  48437 non-null  object
 1   type      48437 non-null  object
 2   make      48437 non-null  object
 3   model     48437 non-null  object
 4   colour    48437 non-null  object
 5   fuel      48437 non-null  object
 6   state     48437 non-null  object
dtypes: object(7)
memory usage: 3.0+ MB


,date_reg,type,make,model,colour,fuel,state
0,2025-01-01,motokar,BYD,SEAL,white,electric,Rakan Niaga
1,2025-01-01,window_van,CAM,PLACER-X,yellow,greendiesel,Johor
2,2025-01-01,jip,CHERY,JAECOO J7,green,petrol,Rakan Niaga
3,2025-01-01,jip,CHERY,JAECOO J7,silver,petrol,Rakan Niaga
4,2025-01-01,jip,CHERY,TIGGO,grey,petrol,Rakan Niaga


In [85]:
#easy way to check on their columns
print("Cars2025 columns:", cars_df.columns.tolist())
print("FuelConsumption columns:", fuel_df.columns.tolist())


Cars2025 columns: ['date_reg', 'type', 'make', 'model', 'colour', 'fuel', 'state']
FuelConsumption columns: ['modelyear', 'make', 'model', 'vehicleclass', 'enginesize', 'cylinders', 'transmission', 'fueltype', 'fuelconsumption_city', 'fuelconsumption_hwy', 'fuelconsumption_comb', 'fuelconsumption_comb_mpg', 'co2emissions']


In [56]:
#Check again for their missing value
print(cars_df.isnull().sum())
print(fuel_df.isnull().sum())

#Remove duplicate rows
cars_df = cars_df.drop_duplicates()
fuel_df = fuel_df.drop_duplicates()

date_reg    0
type        0
make        0
model       0
colour      0
fuel        0
state       0
dtype: int64
modelyear                   0
make                        0
model                       0
vehicleclass                0
enginesize                  0
cylinders                   0
transmission                0
fueltype                    0
fuelconsumption_city        0
fuelconsumption_hwy         0
fuelconsumption_comb        0
fuelconsumption_comb_mpg    0
co2emissions                0
dtype: int64


In [50]:
#Standardizing
#Remove spaces, symbols and change string to lowercase
cars_df.columns = cars_df.columns.str.strip().str.lower().str.replace(' ', '_')
fuel_df.columns = fuel_df.columns.str.strip().str.lower().str.replace(' ', '_')

#rename the maker to make to standardize
cars_df.rename(columns={'maker': 'make'}, inplace=True)

#there are same attributes in both datasets, change them to be identical
cars_df['make'] = cars_df['make'].str.strip().str.upper()
cars_df['model'] = cars_df['model'].str.strip().str.upper()
fuel_df['make'] = fuel_df['make'].str.strip().str.upper()
fuel_df['model'] = fuel_df['model'].str.strip().str.upper()

In [63]:
#Remove outliers from fuelConsumption data
fuel_cleaned = fuel_df[
    (fuel_df['fuelconsumption_comb'] < 30) &
    (fuel_df['co2emissions'] < 600)
]


In [69]:
print(f"Cleaned cars_2025 shape: {cars_df.shape}")
print(f"Cleaned fuel_consumption shape: {fuel_cleaned.shape}")

Cleaned cars_2025 shape: (48437, 7)
Cleaned fuel_consumption shape: (1067, 13)


In [97]:
# Merge datasets on make and model to match registrations with emissions info
merged_data = pd.merge(cars_df,fuel_cleaned,
                        on=['make', 'model'],
                        how='inner'
)

# Show the shape of the merged dataset
print(f"Merged data shape: {merged_data.shape}")

#Easy way to check on their column
print("merged_data columns:")
print(merged_data.columns.tolist())


Merged data shape: (9212, 18)
merged_data columns:
['date_reg', 'type', 'make', 'model', 'colour', 'fuel', 'state', 'modelyear', 'vehicleclass', 'enginesize', 'cylinders', 'transmission', 'fueltype', 'fuelconsumption_city', 'fuelconsumption_hwy', 'fuelconsumption_comb', 'fuelconsumption_comb_mpg', 'co2emissions']


In [101]:
# Show the first 5 rows to see all columns combined
merged_data.info()
merged_data.describe()
merged_data.head()

import os

if os.path.exists("cleaned_data.csv"):
    os.remove("cleaned_data.csv")

    merged_data.to_csv('merged_data.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9212 entries, 0 to 9211
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date_reg                  9212 non-null   object 
 1   type                      9212 non-null   object 
 2   make                      9212 non-null   object 
 3   model                     9212 non-null   object 
 4   colour                    9212 non-null   object 
 5   fuel                      9212 non-null   object 
 6   state                     9212 non-null   object 
 7   modelyear                 9212 non-null   int64  
 8   vehicleclass              9212 non-null   object 
 9   enginesize                9212 non-null   float64
 10  cylinders                 9212 non-null   int64  
 11  transmission              9212 non-null   object 
 12  fueltype                  9212 non-null   object 
 13  fuelconsumption_city      9212 non-null   float64
 14  fuelcons

Aggregation

In [34]:
# 1. Average CO2 Emissions by Maker (descending order)
avg_emission_by_maker = merged_df.groupby('maker')['CO2EMISSIONS'].mean().sort_values(ascending=False)
print("Average CO2 Emissions by Maker:")
print(avg_emission_by_maker)
print("\n")

# 2. Average CO2 Emissions by Fuel Type (ascending order)
avg_emission_by_fuel = merged_df.groupby('fuel_clean_x')['CO2EMISSIONS'].mean().sort_values()
print("Average CO2 Emissions by Fuel Type:")
print(avg_emission_by_fuel)
print("\n")

# 3. Count of Unique Car Models per Maker (descending order)
model_counts = merged_df.groupby('maker')['model'].nunique().sort_values(ascending=False)
print("Count of Unique Car Models per Maker:")
print(model_counts)
print("\n")

# 4. Average Engine Size by Maker (descending order)
avg_engine_size = merged_df.groupby('maker')['ENGINESIZE'].mean().sort_values(ascending=False)
print("Average Engine Size by Maker:")
print(avg_engine_size)
print("\n")

# 5. Emission Statistics (min, max, mean, median) by Vehicle Class
emission_stats_by_class = merged_df.groupby('VEHICLECLASS')['CO2EMISSIONS'].agg(['min', 'max', 'mean', 'median'])
print("Emission Statistics by Vehicle Class:")
print(emission_stats_by_class)
print("\n")

# 6. Total Number of Cars Registered per Fuel Type
cars_per_fuel = merged_df['fuel_clean_x'].value_counts()
print("Total Number of Cars Registered per Fuel Type:")
print(cars_per_fuel)
print("\n")

# 7. Aggregate Multiple Metrics per Maker: mean CO2, mean engine size, unique model count
agg_metrics = merged_df.groupby('maker').agg({
    'CO2EMISSIONS': 'mean',
    'ENGINESIZE': 'mean',
    'model': 'nunique'
}).rename(columns={'model':'unique_model_count'}).sort_values('CO2EMISSIONS', ascending=False)

print("Aggregated Metrics per Maker:")
print(agg_metrics)
print("\n")

# 8. Save Aggregated Metrics to CSV (optional)
agg_metrics.to_csv('aggregated_car_emission_stats.csv')
print("Aggregated metrics saved to 'aggregated_car_emission_stats.csv'")


Average CO2 Emissions by Maker:
maker
rolls royce     376.333333
aston martin    359.000000
bentley         348.600000
maserati        347.000000
nissan          297.000000
porsche         281.244635
dodge           279.500000
audi            272.281250
ford            266.568627
volvo           257.320388
volkswagen      255.923963
hyundai         249.205128
kia             245.193548
jaguar          242.000000
jeep            230.000000
land rover      225.000000
chevrolet       213.833333
subaru          208.500000
mazda           196.610419
mitsubishi      190.000000
honda           178.536755
toyota          174.318476
Name: CO2EMISSIONS, dtype: float64


Average CO2 Emissions by Fuel Type:
fuel_clean_x
gasoline    196.815064
hybrid      200.969723
other       221.150000
Name: CO2EMISSIONS, dtype: float64


Count of Unique Car Models per Maker:
maker
audi            8
toyota          5
porsche         5
honda           4
volkswagen      3
mazda           3
volvo           2
bentle